In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Total transactions: {len(dataset):,}")

README.md:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): reconstructing file:   0%|          |  0.00B /  138MB            

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): downloading bytes:           |  0.00B            

...

Cifer-Fraud-Detection-Dataset-AF-part-8-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Total transactions: 21,000,000


In [3]:
dataset = dataset.select(range(min(1_000_000, len(dataset))))
print(f"Working with: {len(dataset):,} transactions")

Working with: 1,000,000 transactions


In [4]:
fraud_data = dataset.filter(lambda x: x['isFraud'] == 1)
non_fraud_data = dataset.filter(lambda x: x['isFraud'] == 0)

print(f"Fraud transactions: {len(fraud_data):,}")
print(f"Non-fraud transactions: {len(non_fraud_data):,}")

Filter:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Fraud transactions: 1,313
Non-fraud transactions: 998,687


In [5]:
fraud_sample = fraud_data.shuffle(seed=42).select(range(500))
non_fraud_sample = non_fraud_data.shuffle(seed=42).select(range(500))

In [6]:
from datasets import concatenate_datasets

balanced_dataset = concatenate_datasets([fraud_sample, non_fraud_sample])
balanced_dataset = balanced_dataset.shuffle(seed=42)

print(f"Final training set: {len(balanced_dataset)} transactions")

Final training set: 1000 transactions


In [7]:
def create_conversation(example):
    txn_type = example['type']
    amount = example['amount']
    old_orig = example['oldbalanceOrg']
    new_orig = example['newbalanceOrig']
    old_dest = example['oldbalanceDest']
    new_dest = example['newbalanceDest']

    user_msg = f"""Analyze this transaction for fraud risk:
- Type: {txn_type}
- Amount: ${amount:,.2f}
- Sender Balance Before: ${old_orig:,.2f}
- Sender Balance After: ${new_orig:,.2f}
- Recipient Balance Before: ${old_dest:,.2f}
- Recipient Balance After: ${new_dest:,.2f}"""

    if example['isFraud'] == 1:
        asst_msg = "HIGH"
    else:
        asst_msg = "LOW"

    return {"messages": [
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": asst_msg}
    ]}

In [8]:
train_dataset = balanced_dataset.map(create_conversation)
print(f"Created {len(train_dataset)} training conversations")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Created 1000 training conversations


In [10]:
print(train_dataset[1]['messages'])

[{'content': 'Analyze this transaction for fraud risk:\n- Type: CASH_IN\n- Amount: $117,361.76\n- Sender Balance Before: $6,375,305.83\n- Sender Balance After: $21,076,741.69\n- Recipient Balance Before: $3,284,841.99\n- Recipient Balance After: $2,463,933.03', 'role': 'user'}, {'content': 'LOW', 'role': 'assistant'}]


In [11]:
import json

data = train_dataset.to_list()

with open("fraud_detection_conversations.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved {len(data)} conversations")

Saved 1000 conversations
